<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# List All FABRIC Resources

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**Welcome!** This notebook shows you how to query and explore the resources available across all FABRIC testbed sites. Understanding resource availability is essential for planning experiments -- you need to know which sites have the compute, memory, storage, networking, and accelerator capacity your experiment requires.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. List all FABRIC sites and their available resources using `list_sites()`
2. Select specific fields to display (e.g., cores, RAM, disk, NICs)
3. Retrieve resource data in multiple output formats: Pandas DataFrame, text, JSON, and Python list
4. Query resource availability for a **future time window** (advanced scheduling)
5. List per-host resources using `list_hosts()`
6. List facility ports and network links

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you **must** complete the environment setup:

1. Run the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook to create your `fabric_rc` and `ssh_config` files
2. Have a valid FABRIC account with an active project

**Tip:** This notebook does **not** create any slices or consume resources. It only queries the testbed for current availability, so you can safely run it at any time.

For full API details, see the official documentation:
- [Resources](https://fabric-fablib.readthedocs.io/en/latest/resources.html)
- [Site](https://fabric-fablib.readthedocs.io/en/latest/site.html)
- [Host](https://fabric-fablib.readthedocs.io/en/latest/site.html#fabrictestbed_extensions.fablib.site.Host)

</div>

## Background: FABRIC Resource Hierarchy

FABRIC organizes resources in a hierarchy. Understanding this hierarchy helps you target the right level of detail when querying availability.

```
FABRIC Testbed
  +-- Site (e.g., RENC, TACC, UCSD, ...)
  |     +-- Host (physical server within a site)
  |     |     +-- Cores, RAM, Disk
  |     |     +-- GPUs (T4, RTX6000, A30, A40)
  |     |     +-- SmartNICs (ConnectX-5, ConnectX-6)
  |     |     +-- FPGAs (U280)
  |     +-- Facility Ports (external network connections)
  |     +-- Network Links (inter-site connections)
  +-- Site ...
```

- **`list_sites()`** gives you a per-site summary of total and available resources
- **`list_hosts()`** drills down to individual physical servers within each site
- **`list_facility_ports()`** shows external network connection points
- **`list_links()`** shows inter-site network links and their capacities

---

## Step 1: Import FABlib and Verify Configuration

Every FABRIC notebook starts by importing the **FABlib** library and creating a `FablibManager` instance. The `show_config()` call prints your current configuration so you can verify that tokens, keys, and paths are set correctly.

In [ ]:
# Import the FABlib library
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# Create a FABlib manager instance -- this connects to the FABRIC control framework
fablib = fablib_manager()

# Display current configuration (tokens, keys, project info)
fablib.show_config();

## Step 2: List Available Resources by Site

By default, `list_sites()` prints a table of all FABRIC sites. In Jupyter, this renders as a Pandas DataFrame. Outside Jupyter, it falls back to a tabular text string.

The table shows each site's **total** and **available** resources including cores, RAM, disk, NICs, GPUs, and FPGAs.

In [ ]:
# List all FABRIC sites with their available resources
# The returned value is a Pandas DataFrame (in Jupyter) that you can further manipulate
output_table = fablib.list_sites()

### Understanding Column Names

By default, `list_sites()` uses human-readable column headers with spaces (e.g., "Cores Available"). To see the **programmatic field names** that you will use when filtering or selecting columns, pass `pretty_names=False`.

In [ ]:
# Display the same table but with programmatic field names (no spaces)
# These are the names you use in the 'fields' parameter
output_table = fablib.list_sites(pretty_names=False)

## Step 3: Select Specific Fields

When you only care about certain resource types, you can pass a `fields` list to limit the displayed columns. This makes the output much more readable, especially in narrow terminal windows.

<div class="fab-warning">

**Tip:** Use `pretty_names=False` (Step 2 above) to discover the exact field names you can pass in the `fields` list.

</div>

In [ ]:
# Define a list of fields to display -- only show site name and key available resources
fields=['name','cores_available','ram_available','disk_available','nic_basic_available']

In [ ]:
# List sites showing only the selected fields
output_table = fablib.list_sites(fields=fields)

## Step 4: Output Formats

FABlib supports multiple output formats so you can integrate resource data into scripts, APIs, or analysis pipelines. The `output` parameter controls the format.

| `output` Value | Returns | Best For |
|---|---|---|
| `'pandas'` (default in Jupyter) | Pandas DataFrame | Interactive analysis, filtering, plotting |
| `'text'` (default outside Jupyter) | Tabular text string | Terminal display, logging |
| `'json'` | JSON string | REST APIs, data export |
| `'list'` | Python `list[dict]` | Programmatic iteration |

### Output as Pandas DataFrame

In Jupyter, this is the same as the default. The returned `output_table` is a Pandas DataFrame that you can further filter, sort, or plot.

In [ ]:
# Explicitly request Pandas DataFrame output with selected fields
output_table = fablib.list_sites(output='pandas',fields=fields)

### Output as Tabular Text

Useful for scripts or environments without Jupyter. The returned value is a formatted text string.

In [ ]:
# Request plain-text tabular output
output_table = fablib.list_sites(output='text',fields=fields)

### Output as JSON String

Returns a JSON string representation of all sites. JSON output always contains **all fields** regardless of the `fields` parameter.

In [ ]:
# Request JSON output -- always includes all fields
output_json = fablib.list_sites(output='json')

### Output as Python List of Dictionaries

Returns a Python `list[dict]` where each dictionary represents one site. This is the most convenient format for programmatic iteration. Like JSON, it always contains **all fields**.

In [ ]:
# Request Python list output -- always includes all fields
output_list = fablib.list_sites(output='list')

### Iterating Over the List Output

When you set `quiet=True`, the table is not printed automatically. You can then loop through the list and extract exactly the fields you need.

In [ ]:
# Get list output without printing the table (quiet=True suppresses display)
output_list = fablib.list_sites(output='list', quiet=True)

# Iterate over each site dictionary and print selected fields
for site in output_list:
    print(f"Site: {site['name']}, {site['cores_available']}, {site['ram_available']}, {site['disk_available']}, {site['nic_basic_available']}")

## Step 5: List Resources for a Future Time Window

By default, `list_sites()` shows **current** resource availability. You can also check what will be available during a **future time window** by passing `start` and `end` parameters. This is essential for advanced scheduling -- reserving resources for a specific time in the future.

<div class="fab-warning">

**Tip:** You do not need to specify an `end` date. If omitted, the query checks availability starting from the `start` time onward.

</div>

In [ ]:
from datetime import datetime
from datetime import timezone
from datetime import timedelta

# Define a future time window: starting tomorrow, lasting 1 day
start = (datetime.now(timezone.utc) + timedelta(days=1))
end = start + timedelta(days=1)

In [ ]:
# Query resource availability for the future time window
output_table = fablib.list_sites(start=start, end=end)

### Combining Time Windows with Field Filtering

You can combine `start`/`end` with `fields` to focus on specific resources during a future time window. For example, checking SmartNIC availability for tomorrow.

In [ ]:
# Check SmartNIC availability for the future time window
fields=['name','cores_available','ram_available','disk_available','nic_connectx_6_available', 'nic_connectx_5_available']
output_table = fablib.list_sites(start=start, end=end, fields=fields)

In [ ]:
# Add an example for filtered output

## Step 6: List Resources by Host

While `list_sites()` gives a per-site summary, `list_hosts()` drills down to individual **physical servers** within each site. This is useful when you need to ensure your VM lands on a host with specific hardware (e.g., a particular GPU model).

<div class="fab-warning">

**Tip:** You can also pass `start` and `end` parameters to `list_hosts()` to check host-level availability for a future time window:

```python
from datetime import datetime, timezone, timedelta

start = (datetime.now(timezone.utc) + timedelta(days=1))
end = start + timedelta(days=1)
output_table = fablib.list_hosts(start=start, end=end)
```

</div>

In [ ]:
# List all hosts across all sites with their current resource availability
output_table = fablib.list_hosts()

## Step 7: List Facility Ports

**Facility ports** are external network connection points at FABRIC sites that allow connectivity to resources outside the FABRIC testbed (e.g., campus networks, cloud providers, or other research infrastructure).

All the output format options (`output`, `fields`, `start`/`end`) described above also apply to `list_facility_ports()`.

<div class="fab-warning">

**Tip:** Check facility port availability for a future time window:

```python
from datetime import datetime, timezone, timedelta

start = (datetime.now(timezone.utc) + timedelta(days=1))
end = start + timedelta(days=1)
output_table = fablib.list_facility_ports(start=start, end=end)
```

</div>

In [ ]:
# List all facility ports and their availability
output_table = fablib.list_facility_ports()

## Step 8: List Available Network Links

Network links connect FABRIC sites to each other. Listing links shows you the available bandwidth, allocated capacity, and network layer for each inter-site connection. This information helps you choose appropriate paths when designing multi-site experiments.

In [ ]:
# List all inter-site network links with capacity information
fablib.list_links();

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| `show_config()` shows missing values | Environment not configured | Run [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) |
| Empty table from `list_sites()` | Token expired or network issue | Re-authenticate and check your internet connection |
| Unknown field name in `fields` list | Typo or field does not exist | Run `list_sites(pretty_names=False)` to see all valid field names |
| `list_sites(start=..., end=...)` returns no results | No resources available in that time window | Try a wider time window or fewer resource constraints |
| `list_hosts()` is slow | Large number of hosts being queried | This is normal; results are cached for subsequent calls |

## FABlib API Reference

The following FABlib methods were used in this notebook:

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.show_config()` | Display current FABlib configuration | [show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config) |
| `fablib.list_sites()` | List all sites and available resources | [list_sites](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.list_sites) |
| `fablib.list_hosts()` | List all hosts and per-host resources | [list_hosts](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.list_hosts) |
| `fablib.list_facility_ports()` | List external network connection points | [list_facility_ports](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.list_facility_ports) |
| `fablib.list_links()` | List inter-site network links | [list_links](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.list_links) |

## What's Next?

Now that you know how to explore FABRIC resources, try these related notebooks:

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **Filter Sites** | [filter_sites_by_available_resources](./filter_sites_by_available_resources.ipynb) | Use lambda functions to find sites matching specific criteria |
| **Create a Slice** | [hello_fabric](../hello_fabric/hello_fabric.ipynb) | Create your first experiment on FABRIC |
| **Customize Nodes** | [customizing_nodes](../customizing_nodes/customizing_nodes.ipynb) | Set site, cores, RAM, disk, and OS image |
| **Advanced Scheduling** | [advanced_scheduling_slice](../create_slice/advanced_scheduling_slice.ipynb) | Reserve resources for a future time window |
| **GPUs** | [fabric_gpu](../fabric_all_gpus/fabric_gpu.ipynb) | Use NVIDIA GPUs (T4, RTX6000, A30, A40) |